In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
path = '/content/drive/MyDrive/HydroAI/data/raw'
print(os.listdir(path))

['HydroAI _Week2_DataCleaning.ipynb', 'Aqueduct40_baseline_annual_y2023m07d05.csv', 'im3_open_source_data_center_atlas_v2026.02.09.csv']


# Step 3- Load Aqueduct Data (9 columns only)

**Why these 9 columns?**

The WRI Aqueduct 4.0 dataset has 271 columns covering 13 different water risk
indicators. We only load 9 columns because HydroAI only needs two things from
this dataset:

**Location identifiers** (to match data center coordinates to the right region):
- string_id — unique ID for each geographic sub-region
- gid_0 — country code (e.g. "USA")
- name_0 — country name (e.g. "United States")
- gid_1 — state/province code
- name_1 — state/province name (e.g. "California")
- area_km2 — size of the region in square kilometers

**Baseline water stress indicators** (our core metric):
- bws_raw — raw water stress number
- bws_score — normalized score 0-5 (0 = low, 5 = extremely high)
- bws_cat — category code
- bws_label — human readable label (Low / Medium / High / Extremely High)

The other 262 columns cover flood risk, groundwater depletion, and seasonal
variability — real water risk indicators, but outside HydroAI's defined scope.

In [ ]:
import pandas as pd
columns_needed =['string_id', 'gid_0', 'name_0', 'gid_1', 'name_1', 'area_km2', 'bws_raw', 'bws_score', 'bws_cat', 'bws_label']
df = pd.read_csv('/content/drive/MyDrive/HydroAI/data/raw/Aqueduct40_baseline_annual_y2023m07d05.csv', usecols=columns_needed)
print(f"Total rows loaded: {len(df)}")
print(df.head())


Total rows loaded: 68510
              string_id     gid_1 gid_0 name_0      name_1     area_km2  \
0  111011-EGY.11_1-3365  EGY.11_1   EGY  Egypt  Al Qahirah     4.223754   
1  111011-EGY.15_1-3365  EGY.15_1   EGY  Egypt   As Suways  1846.012343   
2  111011-EGY.15_1-None  EGY.15_1   EGY  Egypt   As Suways    30.526067   
3      111011-None-3365     -9999   NaN    NaN         NaN     0.742712   
4      111011-None-None     -9999   NaN    NaN         NaN    13.430995   

   bws_raw  bws_score  bws_cat              bws_label  
0   9999.0        5.0      4.0  Extremely High (>80%)  
1   9999.0        5.0      4.0  Extremely High (>80%)  
2   9999.0        5.0      4.0  Extremely High (>80%)  
3   9999.0        5.0      4.0  Extremely High (>80%)  
4   9999.0        5.0      4.0  Extremely High (>80%)  


## Step 4 — Filter to US Only

We filter the global dataset (68,510 rows covering all countries) down to
US rows only, since HydroAI focuses on US data centers using the IM3 Atlas.

We filter using the gid_0 column which contains country codes — 'USA' for
United States. Every non-US row gets dropped, giving us a much smaller,
focused dataset to work with.

In [ ]:
df_us = df[df['gid_0']== 'USA']
print(f"Rows after filtering to USA: {len(df_us)}")
print(df_us.head())

Rows after filtering to USA: 3433
                 string_id    gid_1 gid_0         name_0  name_1    area_km2  \
29007  355000-USA.2_1-1047  USA.2_1   USA  United States  Alaska  781.766461   
29008  355000-USA.2_1-1064  USA.2_1   USA  United States  Alaska  206.480664   
29009  355000-USA.2_1-1079  USA.2_1   USA  United States  Alaska  254.894601   
29010  355000-USA.2_1-1083  USA.2_1   USA  United States  Alaska  216.348231   
29011  355000-USA.2_1-1085  USA.2_1   USA  United States  Alaska   60.852730   

        bws_raw  bws_score  bws_cat   bws_label  
29007  0.000023        0.0      0.0  Low (<10%)  
29008  0.000023        0.0      0.0  Low (<10%)  
29009  0.000023        0.0      0.0  Low (<10%)  
29010  0.000023        0.0      0.0  Low (<10%)  
29011  0.000023        0.0      0.0  Low (<10%)  


## Step 5 — Replace Missing Values (-9999 → NaN)

WRI uses -9999 as a placeholder meaning "no data available" — it is not
a real water stress score. If we leave -9999 in the dataset it would corrupt
any calculations later (e.g. averaging real scores with a fake -9999 mixed in).

We replace every -9999 with NaN (Not a Number) — Python's standard way of
marking a value as missing, which pandas automatically excludes from
calculations.

In [ ]:
import numpy as np

df_us = df_us.replace(-9999, np.nan)

missing_bws = df_us['bws_score'].isna().sum()
print(f"Rows missing a bws_score: {missing_bws} out of {len(df_us)}")

Rows missing a bws_score: 127 out of 3433


## Step 6 — Drop Rows With No Water Stress Score

127 out of 3,433 US rows (3.7%) are missing a bws_score — WRI could not
calculate a reliable water stress figure for these sub-regions.

Since bws_score is HydroAI's core indicator, a row without it is unusable
for our analysis. We drop these 127 rows, leaving us with a clean, complete
dataset where every row has a real, usable water stress score.

In [ ]:
df_clean = df_us.dropna(subset=['bws_score'])
print(f"Final clean Rows: {len(df_clean)}")
print(df_clean.head())

Final clean Rows: 3306
                 string_id    gid_1 gid_0         name_0  name_1    area_km2  \
29007  355000-USA.2_1-1047  USA.2_1   USA  United States  Alaska  781.766461   
29008  355000-USA.2_1-1064  USA.2_1   USA  United States  Alaska  206.480664   
29009  355000-USA.2_1-1079  USA.2_1   USA  United States  Alaska  254.894601   
29010  355000-USA.2_1-1083  USA.2_1   USA  United States  Alaska  216.348231   
29011  355000-USA.2_1-1085  USA.2_1   USA  United States  Alaska   60.852730   

        bws_raw  bws_score  bws_cat   bws_label  
29007  0.000023        0.0      0.0  Low (<10%)  
29008  0.000023        0.0      0.0  Low (<10%)  
29009  0.000023        0.0      0.0  Low (<10%)  
29010  0.000023        0.0      0.0  Low (<10%)  
29011  0.000023        0.0      0.0  Low (<10%)  


## Step 7 — Save Cleaned Dataset to Google Drive

We save the cleaned dataset to the processed folder in Google Drive.
This keeps our raw and cleaned data separate — good research practice.
Raw data stays untouched in data/raw, cleaned output goes to data/processed.

In [ ]:
output_path = '/content/drive/MyDrive/HydroAI/data/processed/aqueduct_water_stress_usa_2023.csv'
df_clean.to_csv(output_path , index=False)
print("Saved successfully!")
print(f"final dataset shape: {df_clean.shape}")

Saved successfully!
final dataset shape: (3306, 10)


## Step 8 — Final Summary

Quick overview of what we accomplished in this cleaning notebook.

In [ ]:
print("== HydroAI Aqueduct Data Cleaning Summary ==")
print(f"Original dataset shape: {df.shape}")
print(f"Rows after filtering to USA: {len(df_us)}")
print(f"Rows missinf bws_score:{missing_bws} out of {len(df_us)}")
print(f"Final clean Rows: {len(df_clean)}")
print(f"Columns Kept : {list(df_clean.columns)}")
print(f"Water Stress labels available: {df_clean['bws_label'].unique()}")



== HydroAI Aqueduct Data Cleaning Summary ==
Original dataset shape: (68510, 10)
Rows after filtering to USA: 3433
Rows missinf bws_score:127 out of 3433
Final clean Rows: 3306
Columns Kept : ['string_id', 'gid_1', 'gid_0', 'name_0', 'name_1', 'area_km2', 'bws_raw', 'bws_score', 'bws_cat', 'bws_label']
Water Stress labels available: ['Low (<10%)' 'Medium - High (20-40%)' 'Low - Medium (10-20%)'
 'Extremely High (>80%)' 'High (40-80%)' 'Arid and Low Water Use']


## Step 9 — Load IM3 Data Center Atlas

This dataset contains 1,479 data center facilities across the US, with exact
coordinates (lon/lat) for each. We filter to our 4 target companies — Google,
Microsoft, Amazon, Meta — since these are the providers in our project's scope.

519 rows have no operator listed (likely smaller or unbranded facilities from
OpenStreetMap) — these fall outside our scope and will be excluded by the filter.

In [ ]:
import pandas as pd

df_dc = pd.read_csv('/content/drive/MyDrive/HydroAI/data/raw/im3_open_source_data_center_atlas_v2026.02.09.csv')

def classify_company(operator):
  if pd.isna(operator): return None
  op = operator.lower()
  if 'amazon' in op: return 'amazon'
  if 'google' in op: return 'google'
  if 'microsoft' in op: return 'microsoft'
  if 'meta' in op: return 'meta'
  else : return None

df_dc['company'] = df_dc['operator'].apply(classify_company)
df_dc_filtered= df_dc[df_dc['company'].notna()]

print(f"Total facilities loaded: {len(df_dc)}")
print(f"Facilities after filtering to target companies: {len(df_dc_filtered)}")
print(df_dc_filtered['company'].value_counts())



Total facilities loaded: 1479
Facilities after filtering to target companies: 415
company
amazon       213
google        76
meta          66
microsoft     60
Name: count, dtype: int64


Solving error that caused the above code to not function:
Found error : name of the file

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/HydroAI/data/raw'))

['HydroAI _Week2_DataCleaning.ipynb', 'Aqueduct40_baseline_annual_y2023m07d05.csv', 'im3_open_source_data_center_atlas_v2026.02.09.csv']


In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/HydroAI'))
print(os.path.exists('/content/drive/MyDrive/HydroAI/data'))
print(os.path.exists('/content/drive/MyDrive/HydroAI/data/raw'))

True
True
True


## Step 10 — Verify Location Data Quality

Before saving our filtered dataset, we confirm every facility has usable
coordinates. Since HydroAI's geospatial layer depends entirely on matching
each data center's lat/lon to a WRI Aqueduct water stress region, any
missing coordinates here would mean that facility can't be mapped later.

We also preview company, name, and coordinates together as a final visual
check that the data looks correct before saving.

In [ ]:
print("Missing lon/lat coordinates in filtered data:" , df_dc_filtered[['lon','lat']].isna().any(axis=1).sum())
print(df_dc_filtered[['company','name', 'lon','lat' ]].head(10))

Missing lon/lat coordinates in filtered data: 0
      company                         name         lon        lat
1      google           Google Data Center  -81.546515  35.894738
2   microsoft             Project Alluvion  -93.711830  41.515967
5        meta     Meta Henrico Data Center  -77.236455  37.482763
8        meta  Meta Huntsville Data Center  -86.628317  34.844245
10  microsoft      Microsoft Data Center 1  -77.582379  38.802411
11       meta   Meta Prineville Datacenter -120.886382  44.298443
40     amazon                 Amazon IAD11  -77.541111  38.790500
41     amazon                 Amazon IAD24  -77.540006  38.790350
42     amazon                  Amazon IAD7  -77.538277  38.791204
45     amazon                 Amazon IAD10  -77.432876  39.021646


## Step 11 — Save Cleaned IM3 Dataset to Google Drive

Saving our filtered dataset (415 facilities across Amazon, Google, Meta,
and Microsoft) to the processed folder, alongside our cleaned Aqueduct data.

In [ ]:
output_path = '/content/drive/MyDrive/HydroAI/data/processed/im3_data_centers_filtered.csv'
df_dc_filtered.to_csv(output_path , index=False)
print("Saved successfully!")
print(f"Final dataset shape: {df_dc_filtered.shape}")

Saved successfully!
Final dataset shape: (415, 14)
